**Import Packages**

In [1]:
from datetime import datetime
from pyhdf.SD import SD, SDC
import matplotlib.pyplot as plt
import numpy as np 

**Spatial Aggregation Function**

In [3]:
def aggregate_to_1km(data):
    """
    Aggregates 4800x4800 250m data to 1200x1200 1km data
    by taking the mean of each 4x4 block.
    
    Returns:
        np.ndarray: Aggregated 1km image of shape (1200, 1200)
    """
    if data.shape == (4800, 4800):
        # for 4800x4800 data => aggregate 4x4 blocks
        reshaped = data.reshape(1200, 4, 1200, 4)
        aggregated = reshaped.mean(axis=(1, 3))
    elif data.shape == (1200, 1200):
        # No aggregation needed
        aggregated = data
    else:
        raise ValueError(f"Unexpected shape: {data.shape}")
                
    return aggregated # shape: (1200, 1200)

**Process MOD13Q1 - NDVI & EVI (250m)**

In [4]:
def process_mod13q1(file_path):
    """
    Load and process MOD13Q1 NDVI and EVI data from 250m resolution,
    apply scaling, replace fill values, and aggregate to 1km resolution.
    
    Returns:
        Tuple of two NumPy arrays: (ndvi_1km, evi_1km), each shape (1200, 1200)    
    """
    
    # Load the MOD13Q1 tile file
    hdf = SD(file_path, SDC.READ)

    # extract NDVI and EVI 
    # 2D NumPy array of shape (4800, 4800)
    ndvi_data = hdf.select('250m 16 days NDVI')[:]
    evi_data = hdf.select('250m 16 days EVI')[:]

    # Replace fill values (usually -3000)
    # -3000, meaning "no data" or "invalid", replace those with NaN
    ndvi_data = np.where(ndvi_data == -3000, np.nan, ndvi_data)
    evi_data = np.where(evi_data == -3000, np.nan, evi_data)
    
    #print(f"MOD13Q1 NDVI original shape: {ndvi_data.shape}")
    #print(f"MOD13Q1 EVI original shape: {evi_data.shape}")

    # MODIS products like MOD13Q1 store many of their scientific data layers (like NDVI, EVI, reflectance, etc.) as scaled integers in the HDF file.
    # This is done to Save storage (integers are smaller than floats), Maintain precision (avoiding floating-point rounding errors during storage)
    # Apply scale factor (0.0001), converts the raw integer NDVI and EVI values to real physical values in the range -1.0 to 1.0 using the scale factor from the metadata.
    ndvi_scaled = ndvi_data * 0.0001
    evi_scaled = evi_data * 0.0001
    
    # Aggregate to 1km
    ndvi_1km = aggregate_to_1km(ndvi_scaled)
    evi_1km = aggregate_to_1km(evi_scaled)

    hdf.end()
    
    return ndvi_1km, evi_1km

In [ ]:
# Function to visualize features, masks, and FIRMS fire points

def plot_tile_analysis(
    ndvi: np.ndarray, 
    evi: np.ndarray, 
    lst_day: np.ndarray, 
    lst_night: np.ndarray, 
    valid_mask: np.ndarray, 
    fire_mask: np.ndarray, 
    matched_points: list, 
    TARGET_TILE: tuple, 
    start_date: datetime, 
    end_date: datetime
):
    """
    Visualizes the 1km resolution features (NDVI, EVI, LST Day, LST Night),
    the valid data mask, and the fire mask overlaid with FIRMS fire points
    for a specific MODIS tile and time window.

    Args:
        ndvi (np.ndarray): 1km NDVI data (1200, 1200).
        evi (np.ndarray): 1km EVI data (1200, 1200).
        lst_day (np.ndarray): 1km LST Day data in Celsius (1200, 1200).
        lst_night (np.ndarray): 1km LST Night data in Celsius (1200, 1200).
        valid_mask (np.ndarray): Mask of valid (non-NaN) pixels (1200, 1200).
        fire_mask (np.ndarray): Binary mask indicating fire pixels (1200, 1200).
        matched_points (list): List of (row, column) indices of FIRMS fire points.
        TARGET_TILE (tuple): The (h, v) tile coordinates.
        start_date (datetime): Start date of the observation period.
        end_date (datetime): End date of the observation period.
    """

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # NDVI
    im1 = axes[0, 0].imshow(ndvi, cmap='YlGn', vmin=0, vmax=1)
    axes[0, 0].set_title('NDVI (1km resolution)')
    plt.colorbar(im1, ax=axes[0, 0])

    # EVI
    im2 = axes[0, 1].imshow(evi, cmap='YlGn', vmin=0, vmax=1)
    axes[0, 1].set_title('EVI (1km resolution)')
    plt.colorbar(im2, ax=axes[0, 1])

    # LST Day
    im3 = axes[0, 2].imshow(lst_day, cmap='coolwarm', vmin=-10, vmax=40)
    axes[0, 2].set_title('LST Day (°C) (1km resolution)')
    plt.colorbar(im3, ax=axes[0, 2])

    # LST Night
    im4 = axes[1, 0].imshow(lst_night, cmap='coolwarm', vmin=-10, vmax=40)
    axes[1, 0].set_title('LST Night (°C) (1km resolution)')
    plt.colorbar(im4, ax=axes[1, 0])

    # Valid data mask
    im5 = axes[1, 1].imshow(valid_mask, cmap='Greys')
    axes[1, 1].set_title('Valid pixels mask (non-NaN)')

    # Fire mask + overlay fire points
    axes[1, 2].imshow(ndvi, cmap='YlGn', vmin=0, vmax=1)
    axes[1, 2].imshow(valid_mask, cmap='gray', alpha=0.3)
    axes[1, 2].imshow(fire_mask, cmap='Reds', alpha=0.25)

    for i, j in matched_points:
        if 0 <= i < 1200 and 0 <= j < 1200:
            axes[1, 2].plot(j, i, 'ro', markersize=2, alpha=0.6)

    axes[1, 2].set_title('FIRMS Fire Points over NDVI (+mask & fire)')

    # Optional: Draw tile boundary
    axes[1, 2].add_patch(plt.Rectangle((0, 0), 1200, 1200, edgecolor='black', facecolor='none', linewidth=1))

    # Convert TARGET_TILE tuple (h, v) to the string format 'hXXvYY' for the title
    tile_str = f"h{TARGET_TILE[0]:02d}v{TARGET_TILE[1]:02d}"

    plt.suptitle(f'Tile {tile_str} | {start_date.date()} to {end_date.date()}', fontsize=16)
    plt.tight_layout()
    plt.show()

# Use as such:
# plot_tile_analysis(
#     ndvi, evi, lst_day, lst_night, 
#     valid_mask, fire_mask, matched_points, 
#     TARGET_TILE, start_date, end_date
# )